In [ ]:
import os
os.makedirs('/tmp/l3/fusion', exist_ok=True)
open('/tmp/l3/fusion/late_fusion.py','w',encoding='utf-8').write('"""Stage D: late (object-level) fusion of Camera+LiDAR and Camera+Radar detections. No learning; all constants are priors fixed before evaluation.\n\nInput/Output: nuScenes detection-result json (global frame).  Association: Hungarian on BEV centre distance, same class only, class-dependent gate.\nMatched pair : geometry/yaw/attributes weighted toward Camera+LiDAR, velocity = score-weighted mean, score = noisy-OR.\nUnmatched    : kept, score multiplied by a penalty (a single-sensor detection is less trustworthy than an agreed one)."""\nimport argparse, json, sys\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nGATE = {\'car\': 2.5, \'truck\': 3.5, \'bus\': 4.0, \'trailer\': 4.0, \'construction_vehicle\': 4.0, \'pedestrian\': 1.5, \'motorcycle\': 2.0,\n        \'bicycle\': 1.5, \'traffic_cone\': 1.0, \'barrier\': 1.5}\nDEFAULTS = dict(s_min=0.05, w_a=0.7, w_b=0.3, pen_a=0.85, pen_b=0.6, vel=\'mean\', max_boxes=500)\n\n\ndef fuse_sample(A, B, p):\n    A = [b for b in A if b[\'detection_score\'] >= p[\'s_min\']]\n    B = [b for b in B if b[\'detection_score\'] >= p[\'s_min\']]\n    used_b, out = set(), []\n    matched_a = {}\n    for cls in GATE:\n        ia = [i for i, b in enumerate(A) if b[\'detection_name\'] == cls]\n        ib = [j for j, b in enumerate(B) if b[\'detection_name\'] == cls]\n        if not ia or not ib:\n            continue\n        ca = np.array([A[i][\'translation\'][:2] for i in ia]); cb = np.array([B[j][\'translation\'][:2] for j in ib])\n        d = np.linalg.norm(ca[:, None, :] - cb[None, :, :], axis=2) / GATE[cls]\n        cost = np.where(d <= 1.0, d, 1e3)\n        r, c = linear_sum_assignment(cost)\n        for a, b in zip(r, c):\n            if cost[a, b] < 1e3:\n                matched_a[ia[a]] = ib[b]; used_b.add(ib[b])\n    for i, a in enumerate(A):\n        if i in matched_a:\n            b = B[matched_a[i]]\n            sa, sb = a[\'detection_score\'], b[\'detection_score\']\n            wa, wb = p[\'w_a\'] * sa, p[\'w_b\'] * sb\n            t = (wa * np.array(a[\'translation\']) + wb * np.array(b[\'translation\'])) / (wa + wb)\n            if p[\'vel\'] == \'mean\':\n                v = (sa * np.array(a[\'velocity\']) + sb * np.array(b[\'velocity\'])) / (sa + sb)\n            elif p[\'vel\'] == \'a\':\n                v = np.array(a[\'velocity\'])\n            else:\n                v = np.array(b[\'velocity\'])\n            f = dict(a); f.update(translation=t.tolist(), velocity=v.tolist(), detection_score=float(1 - (1 - sa) * (1 - sb)))\n            f[\'_src\'] = \'both\'; f[\'_sa\'] = sa; f[\'_sb\'] = sb\n            out.append(f)\n        else:\n            f = dict(a); f[\'detection_score\'] = a[\'detection_score\'] * p[\'pen_a\']; f[\'_src\'] = \'cl\'; f[\'_sa\'] = a[\'detection_score\']; f[\'_sb\'] = 0.0\n            out.append(f)\n    for j, b in enumerate(B):\n        if j not in used_b:\n            f = dict(b); f[\'detection_score\'] = b[\'detection_score\'] * p[\'pen_b\']; f[\'_src\'] = \'cr\'; f[\'_sa\'] = 0.0; f[\'_sb\'] = b[\'detection_score\']\n            out.append(f)\n    out.sort(key=lambda x: -x[\'detection_score\'])\n    return out[:p[\'max_boxes\']]\n\n\ndef fuse(resA, resB, **kw):\n    p = dict(DEFAULTS); p.update(kw)\n    assert set(resA[\'results\']) == set(resB[\'results\']), \'sample sets differ\'\n    res = {t: fuse_sample(resA[\'results\'][t], resB[\'results\'][t], p) for t in resA[\'results\']}\n    return res, p\n\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--a\', required=True, help=\'Camera+LiDAR results json\'); ap.add_argument(\'--b\', required=True, help=\'Camera+Radar results json\')\n    ap.add_argument(\'--out\', required=True)\n    for k, v in DEFAULTS.items():\n        ap.add_argument(\'--\' + k, type=type(v), default=v)\n    a = ap.parse_args()\n    A = json.load(open(a.a)); B = json.load(open(a.b))\n    res, p = fuse(A, B, **{k: getattr(a, k) for k in DEFAULTS})\n    meta = dict(use_camera=True, use_lidar=True, use_radar=True, use_map=False, use_external=False)\n    sub = dict(meta=meta, results={t: [{k: v for k, v in b.items() if not k.startswith(\'_\')} for b in bs] for t, bs in res.items()})\n    json.dump(sub, open(a.out, \'w\'))\n    json.dump({t: [(b[\'_src\'], b[\'_sa\'], b[\'_sb\']) for b in bs] for t, bs in res.items()}, open(a.out.replace(\'.json\', \'_sources.json\'), \'w\'))\n    n = {s: sum(1 for bs in res.values() for b in bs if b[\'_src\'] == s) for s in (\'both\', \'cl\', \'cr\')}\n    print(\'fused\', sum(len(v) for v in res.values()), \'boxes\', n, \'params\', p)\n')
open('/tmp/l3/fusion_eval.py','w',encoding='utf-8').write('"""Official nuScenes detection eval (mini_val) for single-branch systems and late-fusion variants."""\nimport glob, json, os, sys, time\nsys.path.insert(0, \'/tmp/l3/fusion\')\nfrom late_fusion import fuse\nfrom nuscenes import NuScenes\nfrom nuscenes.eval.detection.config import config_factory\nfrom nuscenes.eval.detection.evaluate import NuScenesEval\n\nOUT = \'/kaggle/working/outputs/fusion_eval\'\nos.makedirs(OUT, exist_ok=True)\nV = None\nfor r, d, f in os.walk(\'/kaggle/input\'):\n    if \'v1.0-mini\' in d:\n        V = r; break\nprint(\'dataroot\', V)\nnusc = NuScenes(version=\'v1.0-mini\', dataroot=V, verbose=False)\ncfg = config_factory(\'detection_cvpr_2019\')\n\n\ndef find(sub):\n    r = [p for p in glob.glob(\'/kaggle/input/**/results_nusc.json\', recursive=True) if sub in p]\n    assert r, \'missing \' + sub\n    return r[0]\n\n\nP = dict(L3D=find(\'L3D_lidar_camera\'), OURB=find(\'our_bevfusion_mini/predictions/final_last\'), CRN=find(\'crn_r18_pretrained\'))\nprint(json.dumps(P, indent=1))\nR = {k: json.load(open(v)) for k, v in P.items()}\nMETA = dict(use_camera=True, use_lidar=True, use_radar=True, use_map=False, use_external=False)\n\n\ndef evaluate(name, results, save=False):\n    sub = dict(meta=META, results={t: [{k: v for k, v in b.items() if not k.startswith(\'_\')} for b in bs] for t, bs in results.items()})\n    d = f\'/tmp/l3/fe/{name}\'; os.makedirs(d, exist_ok=True)\n    p = f\'{d}/results_nusc.json\'; json.dump(sub, open(p, \'w\'))\n    t0 = time.time()\n    NuScenesEval(nusc, config=cfg, result_path=p, eval_set=\'mini_val\', output_dir=d, verbose=False).main(render_curves=False)\n    m = json.load(open(f\'{d}/metrics_summary.json\'))\n    row = dict(name=name, mAP=m[\'mean_ap\'], NDS=m[\'nd_score\'], mATE=m[\'tp_errors\'][\'trans_err\'], mASE=m[\'tp_errors\'][\'scale_err\'],\n               mAOE=m[\'tp_errors\'][\'orient_err\'], mAVE=m[\'tp_errors\'][\'vel_err\'], mAAE=m[\'tp_errors\'][\'attr_err\'], AP2m=m[\'mean_dist_aps\'],\n               n_boxes=sum(len(v) for v in results.values()), secs=round(time.time() - t0, 1))\n    if save:\n        os.makedirs(f\'{OUT}/submissions\', exist_ok=True)\n        json.dump(sub, open(f\'{OUT}/submissions/{name}.json\', \'w\'))\n        json.dump({t: [(b[\'_src\'], b[\'_sa\'], b[\'_sb\']) for b in bs] for t, bs in results.items() if bs and \'_src\' in bs[0]}, open(f\'{OUT}/submissions/{name}_sources.json\', \'w\'))\n    print(f"{name:34s} mAP={row[\'mAP\']:.4f} NDS={row[\'NDS\']:.4f} mATE={row[\'mATE\']:.3f} mAOE={row[\'mAOE\']:.3f} mAVE={row[\'mAVE\']:.3f} boxes={row[\'n_boxes\']} ({row[\'secs\']}s)", flush=True)\n    return row\n\n\nrows = []\nplain = lambda k: {t: v for t, v in R[k][\'results\'].items()}\nrows.append(evaluate(\'CL_official_pretrained\', plain(\'L3D\')))\nrows.append(evaluate(\'CL_ours_trained\', plain(\'OURB\')))\nrows.append(evaluate(\'CR_crn_pretrained\', plain(\'CRN\')))\nfor tag, A in ((\'official\', \'L3D\'), (\'ours\', \'OURB\')):\n    variants = [(\'baseline\', {}, True), (\'no_penalty\', dict(pen_a=1.0, pen_b=1.0), False), (\'vel_from_CL\', dict(vel=\'a\'), False),\n                (\'vel_from_CR\', dict(vel=\'b\'), False), (\'equal_weights\', dict(w_a=0.5, w_b=0.5), False),\n                (\'strong_penalty\', dict(pen_a=0.7, pen_b=0.4), False)]\n    for vn, kw, save in variants:\n        res, p = fuse(R[A], R[\'CRN\'], **kw)\n        rows.append(evaluate(f\'FUSED_{tag}_{vn}\', res, save=save))\njson.dump(rows, open(f\'{OUT}/fusion_eval.json\', \'w\'), indent=2)\nprint(\'FUSION EVAL PASS\', len(rows), \'systems\')\n')
print('written')

In [ ]:
%%bash
set -euo pipefail
export W=/tmp/l3; export MPLBACKEND=Agg
TAR=$(find /kaggle/input -maxdepth 6 -name 'env_bevfusion.tar.*' | head -1); echo "$TAR"
mkdir -p $W/fe /kaggle/working/outputs
case "$TAR" in *.zst) zstd -dc "$TAR" | tar -C $W -xf - envs/bevfusion ;; *.gz) tar -C $W -xzf "$TAR" envs/bevfusion ;; esac
PY=$W/envs/bevfusion/bin/python
$PY -c "import nuscenes, scipy, numpy; print('devkit', nuscenes.__file__, 'scipy', scipy.__version__, 'numpy', numpy.__version__)"
$PY -W ignore $W/fusion_eval.py 2>&1 | tail -40
ls -la /kaggle/working/outputs/fusion_eval /kaggle/working/outputs/fusion_eval/submissions
